# Support Ticket Knowledge Assistant

**Project 17** — Core RAG Projects

Build a system that retrieves similar past support tickets and suggests likely
fixes for new incoming tickets. Learn the practical difference between semantic
similarity and keyword search — and when each approach wins or fails.

**Key Skills:** Ticket similarity search, resolution suggestion, semantic vs keyword
comparison, category-aware retrieval, resolution confidence scoring.

## Project Overview

Support teams handle thousands of tickets. Many tickets describe the same problem
in different words. A knowledge assistant that finds similar past tickets and
surfaces their resolutions can:

1. **Reduce resolution time** — agents see what worked before
2. **Improve consistency** — same problem gets the same fix
3. **Enable self-service** — users find answers without filing a ticket
4. **Train new agents** — learn from resolved ticket history

This notebook builds a **Support Ticket Knowledge Assistant** that:
- Maintains a knowledge base of 30 resolved tickets across 5 categories
- Finds the most similar past tickets for any new incoming ticket
- Suggests likely fixes based on resolution history
- Compares semantic similarity vs keyword search head-to-head
- Explains *why* semantic search works better for natural-language ticket text

### The Core Challenge

Users describe the same problem in many ways:
- "Can't log in" / "Login broken" / "Authentication failing" / "Password not working"
- "App crashes on start" / "Program won't open" / "Fatal error at launch"

**Keyword search** matches exact words — it misses "authentication failing" when
searching for "login broken" because the words don't overlap.

**Semantic search** matches meaning — it understands that "can't log in" and
"authentication failing" describe the same problem.

## Learning Goals

By the end of this notebook you will understand:
- How to build a ticket similarity search system
- The fundamental difference between keyword and semantic matching
- When keyword search outperforms semantic search (and vice versa)
- How to suggest fixes based on similar resolved tickets
- How to evaluate retrieval quality for support ticket use cases
- How category and priority metadata improve search relevance

## Problem Statement

**Scenario:** TechSupport Inc. handles 500+ tickets/day for a SaaS platform.
They have a knowledge base of resolved tickets spanning 5 categories:
Authentication, Billing, Performance, Integration, and Data.

When a new ticket arrives, the agent needs to quickly find similar past tickets
and see what resolution worked. Currently agents search by keyword, which
misses many relevant matches.

**Goal:** Build a similarity search that finds relevant past tickets regardless
of how the problem is described, and suggest the most likely fix with a
confidence score.

## Why This Project Matters

| Metric | Without Assistant | With Assistant |
|--------|------------------|----------------|
| Avg resolution time | 45 min | 15 min |
| First-contact resolution | 40% | 70% |
| Ticket reopens | 25% | 10% |
| Agent training time | 3 months | 1 month |

### Real-world applications
- **Help desks** — IT, HR, facilities support ticket routing
- **Customer support** — SaaS, e-commerce, telecom support
- **Bug tracking** — finding duplicate issues in Jira/GitHub
- **Knowledge bases** — Zendesk, Freshdesk, ServiceNow automations
- **Self-service portals** — users search before filing a ticket

## Environment Setup

In [1]:
import os
import re
import json
import hashlib
import textwrap
import warnings
from collections import defaultdict, Counter
from datetime import datetime, timedelta
import random

import numpy as np
warnings.filterwarnings("ignore")

# ── Optional dependencies ──
USE_CHROMA = False
USE_ST = False

try:
    import chromadb
    USE_CHROMA = True
    print("[OK] chromadb available")
except ImportError:
    print("[INFO] chromadb not available — using TF-IDF fallback")

try:
    from sentence_transformers import SentenceTransformer
    USE_ST = True
    print("[OK] sentence-transformers available")
except ImportError:
    print("[INFO] sentence-transformers not available — using TF-IDF fallback")

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

print("\nSetup complete.")

[OK] chromadb available


[OK] sentence-transformers available

Setup complete.


In [2]:
# ── Configuration ──
TOP_K = 5
SIMILARITY_THRESHOLD = 0.3   # minimum similarity to consider relevant
EMBEDDING_MODEL = "all-MiniLM-L6-v2"
SAVE_DIR = os.path.dirname(os.path.abspath("__file__"))

print(f"Top-K: {TOP_K}, Similarity threshold: {SIMILARITY_THRESHOLD}")

Top-K: 5, Similarity threshold: 0.3


## Support Ticket Knowledge Base

We create 30 resolved support tickets across 5 categories. Each ticket has:
- **Subject** and **description** (how the user reported the issue)
- **Category** and **priority**
- **Resolution** (what fixed the problem)
- **Resolution time** and **status** (all resolved)

The tickets include realistic variation — same problems described differently,
similar symptoms with different root causes, and varying levels of detail.

In [3]:
# ── Resolved ticket knowledge base ──

RESOLVED_TICKETS = [
    # ── Authentication (6 tickets) ──
    {
        "ticket_id": "TK-001",
        "subject": "Cannot log in to dashboard",
        "description": "I am unable to log in to my account. I enter my email and password but it says invalid credentials. I have tried resetting my password but I still cannot get in. This started happening after your last update.",
        "category": "Authentication",
        "priority": "high",
        "resolution": "The user's account was locked after 5 failed login attempts due to the new security policy. Unlocked the account via admin panel and advised the user to reset password using the 'Forgot Password' link. Also confirmed the email address was correct (user was using an old email).",
        "resolution_time_hours": 2,
    },
    {
        "ticket_id": "TK-002",
        "subject": "SSO not working after domain change",
        "description": "We recently changed our company domain from oldcompany.com to newcompany.com. Now Single Sign-On authentication is failing for all our users. We are on the Enterprise plan with SAML SSO configured.",
        "category": "Authentication",
        "priority": "critical",
        "resolution": "Updated the SAML SSO configuration in the admin panel to reflect the new domain. Changed the Entity ID and ACS URL to use newcompany.com. Also updated the IdP metadata XML with the new certificate. SSO was restored after the IdP-side configuration was also updated by the customer's IT team.",
        "resolution_time_hours": 4,
    },
    {
        "ticket_id": "TK-003",
        "subject": "Two-factor authentication codes not working",
        "description": "My 2FA codes from Google Authenticator are being rejected. I have tried multiple times and the codes are always wrong. I need to access my account urgently for a deadline.",
        "category": "Authentication",
        "priority": "high",
        "resolution": "The user's device clock was out of sync, causing TOTP codes to be invalid. Guided the user to enable automatic time sync on their phone (Settings > Date & Time > Automatic). Alternatively, generated backup codes and sent them to the user's verified recovery email as an immediate workaround.",
        "resolution_time_hours": 1,
    },
    {
        "ticket_id": "TK-004",
        "subject": "API key returning 401 unauthorized",
        "description": "Our integration is suddenly getting 401 errors. The API key was working fine until yesterday. We haven't changed anything on our end. This is blocking our production pipeline.",
        "category": "Authentication",
        "priority": "critical",
        "resolution": "The API key had been automatically rotated as part of a scheduled security rotation that the account admin had configured 90 days ago. Generated a new API key and provided it to the customer. Recommended storing keys in a secrets manager and setting up key rotation notifications.",
        "resolution_time_hours": 1,
    },
    {
        "ticket_id": "TK-005",
        "subject": "Password reset email never arrives",
        "description": "I requested a password reset but never received the email. I've checked spam and junk folders. I've tried 3 times over the past hour. My email is john@example.com.",
        "category": "Authentication",
        "priority": "medium",
        "resolution": "The user's email domain had strict DMARC policies that were rejecting our password reset emails. Added the domain to our email allowlist and resent the reset email. Also provided a direct reset link as workaround. Long-term fix: configured DKIM signing for our password reset email sender.",
        "resolution_time_hours": 3,
    },
    {
        "ticket_id": "TK-006",
        "subject": "Account shows as deactivated",
        "description": "When I try to log in, it says my account has been deactivated. I haven't done anything to deactivate it. I need access to my data immediately. I'm a paying customer on the Pro plan.",
        "category": "Authentication",
        "priority": "high",
        "resolution": "The account was auto-deactivated due to 180 days of inactivity (company policy). Reactivated the account, restored all data, and confirmed that the subscription billing was still active. Added an exception to prevent future auto-deactivation for paid accounts.",
        "resolution_time_hours": 2,
    },

    # ── Billing (6 tickets) ──
    {
        "ticket_id": "TK-007",
        "subject": "Charged twice for subscription",
        "description": "I see two charges on my credit card for this month's subscription. I should only be charged once. The amounts are $49.99 each. Please refund the duplicate charge.",
        "category": "Billing",
        "priority": "high",
        "resolution": "Confirmed duplicate charge in the billing system. The second charge was caused by a timeout during payment processing, which triggered a retry. Issued a full refund for the duplicate charge ($49.99). Refund will appear in 5-7 business days. Added idempotency check to prevent future duplicates.",
        "resolution_time_hours": 1,
    },
    {
        "ticket_id": "TK-008",
        "subject": "Cannot update payment method",
        "description": "I am trying to update my credit card on file but the page just shows a spinner and never loads. I've tried in Chrome and Firefox. My current card is expiring next week and I don't want my service to lapse.",
        "category": "Billing",
        "priority": "medium",
        "resolution": "The billing page JavaScript was blocked by the user's ad blocker (uBlock Origin). The Stripe payment iframe was being filtered. Solution: asked user to disable ad blocker for our domain or use the direct billing URL (/settings/billing/update-card). Also filed a bug to add a fallback UI when the iframe is blocked.",
        "resolution_time_hours": 1,
    },
    {
        "ticket_id": "TK-009",
        "subject": "Need to downgrade plan but option is grayed out",
        "description": "I want to downgrade from Enterprise to Pro plan but the downgrade button is grayed out. It shows a tooltip saying 'Cannot downgrade with active add-ons'. I don't know what add-ons it's referring to.",
        "category": "Billing",
        "priority": "low",
        "resolution": "The user had two active add-ons: Advanced Analytics ($20/mo) and Priority Support ($30/mo). These are only available on Enterprise plan. Explained the requirement to remove add-ons before downgrading. Walked the user through Add-ons > Manage > Cancel for each. After cancellation, the downgrade option became available.",
        "resolution_time_hours": 2,
    },
    {
        "ticket_id": "TK-010",
        "subject": "Invoice not showing correct tax amount",
        "description": "Our invoices are not showing the correct VAT amount. We are an EU company and should be charged 20% VAT, but invoices show 0% tax. Our finance team needs corrected invoices for compliance.",
        "category": "Billing",
        "priority": "medium",
        "resolution": "The customer's account had a VAT exemption flag set incorrectly. Updated the tax settings in the billing system to apply 20% VAT for EU customers. Regenerated the last 3 invoices with correct VAT and sent them to the customer's finance team. Also verified the VAT ID on file was valid.",
        "resolution_time_hours": 4,
    },
    {
        "ticket_id": "TK-011",
        "subject": "Requesting refund for unused months",
        "description": "We cancelled our subscription 3 months ago but were still being charged monthly. We want a refund for the 3 months we were charged after cancellation. Total: $149.97.",
        "category": "Billing",
        "priority": "high",
        "resolution": "Investigation showed that the cancellation was submitted but not processed due to a system error in the cancellation workflow. The account remained active. Processed full refund for 3 months ($149.97). Fixed the cancellation bug and confirmed the account is now properly cancelled with data retained for 30 days per policy.",
        "resolution_time_hours": 3,
    },
    {
        "ticket_id": "TK-012",
        "subject": "Usage overage charges unexpectedly high",
        "description": "We received a bill with $500 in overage charges. Our monthly plan is $99. We had no idea we were exceeding our limits. There was no warning or notification before the charges were applied.",
        "category": "Billing",
        "priority": "high",
        "resolution": "The customer exceeded their API call limit (100K/month) by 5x due to a misconfigured automation script running in a loop. Issued a one-time courtesy credit of $250 (50% of overage). Set up usage alerts at 80% and 100% thresholds. Recommended the customer review their script and add rate limiting on their end.",
        "resolution_time_hours": 2,
    },

    # ── Performance (6 tickets) ──
    {
        "ticket_id": "TK-013",
        "subject": "Dashboard loading very slowly",
        "description": "The main dashboard takes over 30 seconds to load. It used to load in 2-3 seconds. This started about a week ago. Other pages seem fine. We have about 50,000 records.",
        "category": "Performance",
        "priority": "high",
        "resolution": "The dashboard query was doing a full table scan on the records table due to a missing index on the created_at column. This became noticeable after the customer's data grew past 40K records. Added a composite index on (account_id, created_at). Dashboard load time dropped from 32s to 1.8s.",
        "resolution_time_hours": 4,
    },
    {
        "ticket_id": "TK-014",
        "subject": "File uploads timing out",
        "description": "Every time I try to upload a file larger than 20MB, the upload times out after about 60 seconds. Smaller files work fine. I'm on a 100Mbps connection so bandwidth shouldn't be the issue.",
        "category": "Performance",
        "priority": "medium",
        "resolution": "The upload endpoint had a 60-second timeout that was too short for large files on the shared upload server. Increased the timeout to 300 seconds for the upload endpoint. Also recommended the customer use the chunked upload API for files over 10MB, which supports resume on failure.",
        "resolution_time_hours": 2,
    },
    {
        "ticket_id": "TK-015",
        "subject": "Search feature returns no results",
        "description": "The search bar in our workspace returns zero results for queries that should have matches. For example, searching for 'quarterly report' returns nothing even though I can see files named 'Q3 Quarterly Report' in the file list. This affects all users in our organization.",
        "category": "Performance",
        "priority": "high",
        "resolution": "The search index for the customer's workspace had become corrupted after a failed index rebuild last Tuesday. Triggered a full search index rebuild for the workspace. Index rebuild completed in 45 minutes and search functionality was restored. Added monitoring alerts for index health checks.",
        "resolution_time_hours": 3,
    },
    {
        "ticket_id": "TK-016",
        "subject": "API response times degraded",
        "description": "Our API calls are taking 5-10 seconds to respond instead of the usual 200-500ms. This is affecting our production application. Started around 2pm today. All endpoints are affected.",
        "category": "Performance",
        "priority": "critical",
        "resolution": "Root cause was a database connection pool exhaustion on the shared infrastructure cluster. A neighboring tenant's bulk import job consumed excessive connections. Isolated the affected tenant, increased the connection pool size, and implemented per-tenant connection limits. Response times returned to normal within 15 minutes of the fix.",
        "resolution_time_hours": 1,
    },
    {
        "ticket_id": "TK-017",
        "subject": "Reports taking too long to generate",
        "description": "Generating our monthly analytics report is now taking over 2 hours. It used to complete in 15 minutes. We haven't changed our report configuration. We need the report for our board meeting tomorrow morning.",
        "category": "Performance",
        "priority": "high",
        "resolution": "The report query was aggregating 18 months of data instead of 1 month due to a date filter bug introduced in last week's release. Fixed the date filter to use the correct month range. Also optimized the aggregation query with pre-computed rollup tables. Report now generates in 8 minutes.",
        "resolution_time_hours": 3,
    },
    {
        "ticket_id": "TK-018",
        "subject": "Mobile app freezing on data sync",
        "description": "The iOS app freezes completely when trying to sync data. I have to force-close and reopen. It happens every time I open the app. I'm on iPhone 14 with iOS 17.3. The app version is 4.2.1.",
        "category": "Performance",
        "priority": "medium",
        "resolution": "A memory leak in the sync module was causing the app to exceed iOS memory limits when syncing more than 5,000 items. The customer had 8,200 items. Released hotfix 4.2.2 that processes sync in batches of 500 items. Also added a pagination parameter to the sync API to support incremental sync.",
        "resolution_time_hours": 8,
    },

    # ── Integration (6 tickets) ──
    {
        "ticket_id": "TK-019",
        "subject": "Slack integration stopped sending notifications",
        "description": "We use the Slack integration to get notified about new uploads. Notifications stopped working 3 days ago. The integration shows as 'Connected' in our settings. Nothing has changed on our Slack workspace.",
        "category": "Integration",
        "priority": "medium",
        "resolution": "The Slack webhook URL had expired because the Slack app was reinstalled in the customer's workspace, generating a new webhook URL. The old URL was still configured in our system. Guided the customer to reconnect the integration (Settings > Integrations > Slack > Reconnect), which generated a fresh webhook URL. Notifications resumed immediately.",
        "resolution_time_hours": 1,
    },
    {
        "ticket_id": "TK-020",
        "subject": "Webhook events not being delivered",
        "description": "Our webhook endpoint is not receiving any events. We configured it to receive file.uploaded and file.deleted events. We've verified our endpoint is accessible and returns 200. The webhook test button in settings works, but real events don't arrive.",
        "category": "Integration",
        "priority": "high",
        "resolution": "The webhook was configured with event types using the old naming convention (file.uploaded) instead of the v2 format (files.upload.completed). Updated the webhook subscription to use v2 event types. Also enabled webhook delivery logs so the customer can debug future issues. Replayed the last 24 hours of missed events.",
        "resolution_time_hours": 2,
    },
    {
        "ticket_id": "TK-021",
        "subject": "Zapier integration creating duplicate records",
        "description": "Our Zapier automation that syncs data from our platform to Google Sheets is creating duplicate rows. Every record appears twice. We have a Zap that triggers on 'New Record' and appends to a Google Sheet.",
        "category": "Integration",
        "priority": "medium",
        "resolution": "The 'New Record' trigger was firing twice because our webhook was sending both a 'record.created' and 'record.synced' event for each new record. The Zapier trigger was subscribed to both events. Adjusted the Zap to filter on event_type='record.created' only. Also updated our Zapier app to not include internal sync events in the New Record trigger.",
        "resolution_time_hours": 2,
    },
    {
        "ticket_id": "TK-022",
        "subject": "Google Drive sync failing with permission error",
        "description": "The Google Drive integration is failing to sync files. It shows 'Permission denied' errors in the sync log. I'm the admin and I've granted all the required permissions in Google. This was working until we migrated to Google Workspace.",
        "category": "Integration",
        "priority": "high",
        "resolution": "The Google Workspace migration changed the OAuth scopes. The old Drive API scopes (drive.file) were deprecated in the new Workspace setup and replaced with new scopes. Guided the customer to revoke and re-authorize the integration via Settings > Integrations > Google Drive > Reconnect. The new OAuth flow requested the updated scopes.",
        "resolution_time_hours": 3,
    },
    {
        "ticket_id": "TK-023",
        "subject": "REST API returning XML instead of JSON",
        "description": "Our API calls are returning XML responses when we expect JSON. We didn't change our Accept headers. This is breaking our parser. Example endpoint: GET /api/v2/records.",
        "category": "Integration",
        "priority": "medium",
        "resolution": "The customer's API gateway (Kong) was overriding the Accept header to application/xml. This was a configuration change made by their DevOps team. Confirmed our API correctly respects the Accept header. Advised the customer to set the Accept header explicitly to application/json in their API gateway config or pass it as a query parameter: ?format=json.",
        "resolution_time_hours": 1,
    },
    {
        "ticket_id": "TK-024",
        "subject": "Custom SSO SAML integration returning invalid response",
        "description": "We are setting up SAML SSO with our Okta identity provider. The integration keeps failing with 'Invalid SAML Response' error. We have followed the setup guide and configured the Entity ID and ACS URL as instructed.",
        "category": "Integration",
        "priority": "high",
        "resolution": "The SAML response was being signed with SHA-1 but our system requires SHA-256 signatures. Updated the Okta application settings to use SHA-256 for both the assertion and response signatures. Also noted that the Name ID format needed to be set to emailAddress instead of persistent. After both changes, SSO authentication worked correctly.",
        "resolution_time_hours": 4,
    },

    # ── Data (6 tickets) ──
    {
        "ticket_id": "TK-025",
        "subject": "Data export contains wrong date format",
        "description": "When I export data to CSV, all dates are in MM/DD/YYYY format but we need them in DD/MM/YYYY format. This is causing issues when we import into our EU-based accounting system. Can this be changed?",
        "category": "Data",
        "priority": "low",
        "resolution": "Added a date format option to the export settings. The customer can now choose between ISO 8601 (YYYY-MM-DD), US format (MM/DD/YYYY), or EU format (DD/MM/YYYY) in Settings > Export > Date Format. Default changed to ISO 8601 for new accounts. Regenerated the export for the customer in DD/MM/YYYY format.",
        "resolution_time_hours": 2,
    },
    {
        "ticket_id": "TK-026",
        "subject": "Imported data showing special characters as garbled text",
        "description": "After importing a CSV file, some fields show garbled characters like 'MÃ¼ller' instead of 'Muller' and 'CafÃ©' instead of 'Cafe'. The original CSV file looks correct when opened in Notepad++.",
        "category": "Data",
        "priority": "medium",
        "resolution": "The CSV file was encoded in UTF-8 with BOM but the import parser was interpreting it as Latin-1. Added UTF-8 BOM detection to the import pipeline. Also added a character encoding selector to the import wizard with auto-detection. Re-ran the import with correct encoding and all special characters displayed properly.",
        "resolution_time_hours": 3,
    },
    {
        "ticket_id": "TK-027",
        "subject": "Bulk delete removed wrong records",
        "description": "I used the bulk delete feature to remove old records from 2022 but it also deleted some 2023 records. Approximately 200 records are missing. We need them restored urgently. The filter I used was 'year = 2022'.",
        "category": "Data",
        "priority": "critical",
        "resolution": "The bulk delete filter had a bug where the year filter used 'created_at < 2023-01-01' which is correct, but also included records that were modified in 2022 regardless of their creation date. Restored all deleted records from the backup (backup was 4 hours old, so some very recent changes were lost). Patched the bulk delete filter to only use created_at. Added a confirmation step showing exact record count before execution.",
        "resolution_time_hours": 6,
    },
    {
        "ticket_id": "TK-028",
        "subject": "Cannot delete records older than 1 year",
        "description": "I'm trying to delete records from 2021 but the delete button is disabled and shows 'Records in retention period cannot be deleted'. We don't have any retention policy set up. Why can't I delete my own data?",
        "category": "Data",
        "priority": "low",
        "resolution": "The account had a default data retention policy of 7 years that was applied automatically to Enterprise accounts for compliance reasons. Explained the retention policy to the customer. Since they confirmed they don't need compliance retention, disabled the default retention policy in their account settings. Delete functionality was restored for all records.",
        "resolution_time_hours": 1,
    },
    {
        "ticket_id": "TK-029",
        "subject": "Data migration from v1 API shows missing fields",
        "description": "We migrated our data from v1 to v2 API format. After migration, several custom fields are showing as empty even though they had values in v1. Specifically, the 'department' and 'cost_center' fields are blank for about 3000 records.",
        "category": "Data",
        "priority": "high",
        "resolution": "The v1-to-v2 migration script had a field mapping issue. The v1 fields 'dept' and 'cost_ctr' were not mapped to the v2 field names 'department' and 'cost_center'. Updated the migration script with correct field mappings and re-ran the migration for the affected 3000 records. All custom field values were restored.",
        "resolution_time_hours": 5,
    },
    {
        "ticket_id": "TK-030",
        "subject": "Automated backup not running",
        "description": "Our daily automated backups haven't run for the past week. The backup schedule shows as active but the last successful backup was 8 days ago. We have critical business data and need assurance it's being backed up.",
        "category": "Data",
        "priority": "critical",
        "resolution": "The backup storage bucket had reached its capacity limit (500GB). Backups were silently failing without notification. Increased the backup storage allocation, ran an immediate full backup, and verified its integrity. Set up backup failure alerts to notify both the customer and our ops team. Added monitoring for backup storage utilization at 80% threshold.",
        "resolution_time_hours": 2,
    },
]

print(f"Created knowledge base of {len(RESOLVED_TICKETS)} resolved tickets\n")
cat_counts = Counter(t["category"] for t in RESOLVED_TICKETS)
for cat, count in cat_counts.most_common():
    print(f"  {cat}: {count} tickets")
pri_counts = Counter(t["priority"] for t in RESOLVED_TICKETS)
print(f"\nBy priority: {dict(pri_counts)}")
avg_res = np.mean([t["resolution_time_hours"] for t in RESOLVED_TICKETS])
print(f"Mean resolution time: {avg_res:.1f} hours")


Created knowledge base of 30 resolved tickets

  Authentication: 6 tickets
  Billing: 6 tickets
  Performance: 6 tickets
  Integration: 6 tickets
  Data: 6 tickets

By priority: {'high': 13, 'critical': 5, 'medium': 9, 'low': 3}
Mean resolution time: 2.6 hours


## Semantic Similarity vs Keyword Search

This is the **central concept** of this project. Let's understand the
fundamental difference with a concrete example.

### The Problem with Keywords

Consider these two tickets:

**Ticket A:** *"My login is broken, it keeps saying wrong password"*
**Ticket B:** *"Authentication failing with invalid credentials error"*

These describe the **exact same problem**. But keyword search fails:

| Query Term | In Ticket A? | In Ticket B? |
|------------|:---:|:---:|
| login | Yes | No |
| broken | Yes | No |
| authentication | No | Yes |
| credentials | No | Yes |
| password | Yes | No |

**Keyword overlap: 0 out of 5 query terms match!** Keyword search would
say these tickets are completely unrelated.

### How Semantic Search Fixes This

Semantic search uses **embedding models** that convert text into numerical
vectors. Similar meanings produce similar vectors, regardless of exact words.

```
"login broken"         → [0.23, 0.87, -0.14, ...]  ─┐
"authentication failing" → [0.21, 0.85, -0.16, ...]  ─┤  cosine similarity ≈ 0.92
                                                        │  (very similar!)
"billing question"     → [-0.45, 0.12, 0.78, ...]  ─┘  cosine similarity ≈ 0.15
                                                        (not similar)
```

### When Each Approach Wins

| Scenario | Winner | Why |
|----------|--------|-----|
| Same problem, different words | Semantic | Captures meaning |
| Exact error codes (ERR-4013) | Keyword | Exact string match |
| Technical jargon (SAML, OAuth) | Tie | Both work with exact terms |
| Vague descriptions | Semantic | Understands intent |
| Typos in query | Semantic | Embeddings handle minor variations |
| Very short queries (2-3 words) | Keyword | Less context for embeddings |

In [4]:
# ── Build ticket index ──

class TicketEntry:
    """A resolved ticket in the knowledge base."""

    def __init__(self, ticket_dict):
        self.ticket_id = ticket_dict["ticket_id"]
        self.subject = ticket_dict["subject"]
        self.description = ticket_dict["description"]
        self.category = ticket_dict["category"]
        self.priority = ticket_dict["priority"]
        self.resolution = ticket_dict["resolution"]
        self.resolution_time = ticket_dict["resolution_time_hours"]

        # Combine subject + description for search
        self.search_text = f"{self.subject}. {self.description}"

    @property
    def metadata(self):
        return {
            "ticket_id": self.ticket_id,
            "category": self.category,
            "priority": self.priority,
            "resolution_time_hours": str(self.resolution_time),
        }

    @property
    def summary(self):
        return f"[{self.ticket_id}] ({self.category}/{self.priority}) {self.subject}"


tickets = [TicketEntry(t) for t in RESOLVED_TICKETS]
print(f"Indexed {len(tickets)} tickets for search")

Indexed 30 tickets for search


In [5]:
# ── Ticket search index ──

class TicketIndex:
    """Search index over resolved support tickets."""

    def __init__(self, tickets):
        self.tickets = tickets
        self.texts = [t.search_text for t in tickets]
        self.ticket_map = {t.ticket_id: t for t in tickets}

        if USE_ST and USE_CHROMA:
            self._build_chroma()
        else:
            self._build_tfidf()

    def _build_chroma(self):
        print("Building index with sentence-transformers + ChromaDB...")
        self.model = SentenceTransformer(EMBEDDING_MODEL)
        self.client = chromadb.Client()
        try:
            self.client.delete_collection("tickets")
        except Exception:
            pass
        self.collection = self.client.create_collection(
            name="tickets", metadata={"hnsw:space": "cosine"}
        )
        embeddings = self.model.encode(self.texts, show_progress_bar=False).tolist()
        ids = [t.ticket_id for t in self.tickets]
        metas = [t.metadata for t in self.tickets]
        self.collection.add(ids=ids, embeddings=embeddings,
                            documents=self.texts, metadatas=metas)
        self.backend = "chroma"
        print(f"Indexed {self.collection.count()} tickets")

    def _build_tfidf(self):
        print("Building TF-IDF fallback index...")
        self.vectorizer = TfidfVectorizer(
            max_features=3000, stop_words="english", ngram_range=(1, 2)
        )
        self.tfidf_matrix = self.vectorizer.fit_transform(self.texts)
        self.backend = "tfidf"
        print(f"Indexed {self.tfidf_matrix.shape[0]} tickets")

    def search(self, query, top_k=TOP_K, category_filter=None):
        if self.backend == "chroma":
            return self._search_chroma(query, top_k, category_filter)
        return self._search_tfidf(query, top_k, category_filter)

    def _search_chroma(self, query, top_k, category_filter):
        where = {"category": category_filter} if category_filter else None
        query_emb = self.model.encode([query]).tolist()
        results = self.collection.query(
            query_embeddings=query_emb,
            n_results=min(top_k, self.collection.count()),
            where=where,
        )
        output = []
        for i, uid in enumerate(results["ids"][0]):
            ticket = self.ticket_map[uid]
            sim = 1.0 - results["distances"][0][i]
            output.append((ticket, sim))
        return output

    def _search_tfidf(self, query, top_k, category_filter):
        candidates = []
        for i, t in enumerate(self.tickets):
            if category_filter and t.category != category_filter:
                continue
            candidates.append(i)
        if not candidates:
            return []
        query_vec = self.vectorizer.transform([query])
        cand_matrix = self.tfidf_matrix[candidates]
        sims = cosine_similarity(query_vec, cand_matrix).flatten()
        top_idx = np.argsort(sims)[::-1][:top_k]
        return [(self.tickets[candidates[i]], float(sims[i])) for i in top_idx]


ticket_index = TicketIndex(tickets)

Building index with sentence-transformers + ChromaDB...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Indexed 30 tickets


## Baseline: Keyword Search

In [6]:
# ── Keyword search baseline ──

def keyword_search(query, tickets, top_k=TOP_K):
    q_terms = set(query.lower().split())
    scored = []
    for t in tickets:
        text_lower = t.search_text.lower()
        hits = sum(1 for term in q_terms if term in text_lower)
        scored.append((t, hits / max(len(q_terms), 1)))
    scored.sort(key=lambda x: x[1], reverse=True)
    return scored[:top_k]

# Test with a semantically similar but lexically different query
test_q = "Authentication failing with invalid credentials"
baseline_r = keyword_search(test_q, tickets, top_k=3)

print(f"Query: \"{test_q}\"\n")
print("Keyword search results:")
for t, score in baseline_r:
    print(f"  {score:.2f} | {t.summary}")
    if score > 0:
        print(f"       Resolution preview: {t.resolution[:80]}...")
    print()

Query: "Authentication failing with invalid credentials"

Keyword search results:
  0.60 | [TK-002] (Authentication/critical) SSO not working after domain change
       Resolution preview: Updated the SAML SSO configuration in the admin panel to reflect the new domain....

  0.60 | [TK-024] (Integration/high) Custom SSO SAML integration returning invalid response
       Resolution preview: The SAML response was being signed with SHA-1 but our system requires SHA-256 si...

  0.40 | [TK-001] (Authentication/high) Cannot log in to dashboard
       Resolution preview: The user's account was locked after 5 failed login attempts due to the new secur...



## Side-by-Side: Semantic vs Keyword

Let's test both approaches on the same queries and see where each wins.

In [7]:
# ── Head-to-head comparison ──

comparison_queries = [
    # Semantically similar but different words
    "Authentication failing with invalid credentials",
    # Should match TK-001 (Cannot log in)

    "Monthly subscription billed two times",
    # Should match TK-007 (Charged twice)

    "Application performance degradation on all API calls",
    # Should match TK-016 (API response times degraded)

    "Google integration authorization problem",
    # Should match TK-022 (Google Drive sync failing)

    "Imported file has corrupted characters",
    # Should match TK-026 (special characters garbled)
]

expected_matches = ["TK-001", "TK-007", "TK-016", "TK-022", "TK-026"]

print(f"{'Query':<50} {'Keyword Top-1':^15} {'Semantic Top-1':^15}")
print("-" * 80)

kw_hits = 0
sem_hits = 0

for q, expected in zip(comparison_queries, expected_matches):
    kw_results = keyword_search(q, tickets, top_k=1)
    sem_results = ticket_index.search(q, top_k=1)

    kw_top = kw_results[0][0].ticket_id if kw_results else "???"
    sem_top = sem_results[0][0].ticket_id if sem_results else "???"

    kw_ok = kw_top == expected
    sem_ok = sem_top == expected
    kw_hits += int(kw_ok)
    sem_hits += int(sem_ok)

    kw_label = f"{kw_top} {'OK' if kw_ok else 'MISS'}"
    sem_label = f"{sem_top} {'OK' if sem_ok else 'MISS'}"

    print(f"  {q[:48]:<48} {kw_label:^15} {sem_label:^15}")

print("-" * 80)
print(f"  {'CORRECT':>48} {kw_hits:^15} {sem_hits:^15}")

Query                                               Keyword Top-1  Semantic Top-1 
--------------------------------------------------------------------------------
  Authentication failing with invalid credentials    TK-002 MISS      TK-001 OK   
  Monthly subscription billed two times              TK-003 MISS      TK-007 OK   
  Application performance degradation on all API c    TK-016 OK       TK-016 OK   
  Google integration authorization problem           TK-021 MISS      TK-022 OK   
  Imported file has corrupted characters              TK-026 OK       TK-026 OK   
--------------------------------------------------------------------------------
                                           CORRECT        2               5       


In [8]:
# ── Deep dive: why keyword fails on paraphrased queries ──

q = "Authentication failing with invalid credentials"
print(f"Query: \"{q}\"\n")

# Keyword analysis
q_terms = set(q.lower().split())
target = next(t for t in tickets if t.ticket_id == "TK-001")
target_terms_found = sum(1 for t in q_terms if t in target.search_text.lower())

print(f"Expected match: TK-001 \"{target.subject}\"")
print(f"Query terms: {q_terms}")
print(f"Terms found in TK-001 text: {target_terms_found}/{len(q_terms)}")
print()

# Show which terms match
for term in sorted(q_terms):
    found = term in target.search_text.lower()
    print(f"  '{term}': {'FOUND' if found else 'NOT FOUND'}")

print(f"\n-> Keyword score: {target_terms_found/len(q_terms):.2f}")
print("-> Semantic search finds this because 'authentication failing' and")
print("   'cannot log in' have the same MEANING, even without shared words.")

Query: "Authentication failing with invalid credentials"

Expected match: TK-001 "Cannot log in to dashboard"
Query terms: {'authentication', 'invalid', 'with', 'failing', 'credentials'}
Terms found in TK-001 text: 2/5

  'authentication': NOT FOUND
  'credentials': FOUND
  'failing': NOT FOUND
  'invalid': FOUND
  'with': NOT FOUND

-> Keyword score: 0.40
-> Semantic search finds this because 'authentication failing' and
   'cannot log in' have the same MEANING, even without shared words.


## Full Support Ticket Knowledge Assistant

In [9]:
# ── Support Ticket Knowledge Assistant ──

class TicketAssistant:
    """Find similar tickets and suggest resolutions."""

    def __init__(self, index):
        self.index = index

    def find_similar(self, subject, description, top_k=TOP_K,
                     category_filter=None, verbose=True):
        """Find similar past tickets and suggest fixes."""
        query = f"{subject}. {description}"
        results = self.index.search(query, top_k=top_k,
                                     category_filter=category_filter)

        # Filter by similarity threshold
        relevant = [(t, s) for t, s in results if s >= SIMILARITY_THRESHOLD]

        # Build resolution suggestion
        suggestion = self._suggest_resolution(relevant)
        confidence = self._confidence(relevant)

        output = {
            "query_subject": subject,
            "similar_tickets": [{
                "ticket_id": t.ticket_id,
                "subject": t.subject,
                "category": t.category,
                "priority": t.priority,
                "similarity": round(s, 3),
                "resolution": t.resolution,
                "resolution_time_hours": t.resolution_time,
            } for t, s in relevant[:top_k]],
            "suggested_resolution": suggestion,
            "confidence": confidence,
            "category_prediction": self._predict_category(relevant),
        }

        if verbose:
            self._display(output)

        return output

    def _suggest_resolution(self, results):
        if not results:
            return "No similar tickets found. This may be a new issue type."

        top_ticket, top_score = results[0]
        suggestion = f"Based on similar ticket {top_ticket.ticket_id} "
        suggestion += f"(similarity: {top_score:.0%}):\n"
        suggestion += f"{top_ticket.resolution}"

        if len(results) > 1:
            other_ids = [t.ticket_id for t, _ in results[1:3]]
            suggestion += f"\n\nAlso review: {', '.join(other_ids)} for related resolutions."

        return suggestion

    def _confidence(self, results):
        if not results:
            return 0.0
        scores = [s for _, s in results]
        top = scores[0]
        avg = np.mean(scores[:3]) if len(scores) >= 3 else np.mean(scores)
        # High confidence when top match is strong and multiple matches agree
        conf = 0.5 * min(top / 0.6, 1.0) + 0.3 * min(avg / 0.4, 1.0)
        # Bonus for multiple relevant matches (consensus)
        if len(results) >= 3:
            conf += 0.1
        if len(results) >= 2 and results[0][0].category == results[1][0].category:
            conf += 0.1  # category agreement
        return round(min(conf, 1.0), 3)

    def _predict_category(self, results):
        if not results:
            return "Unknown"
        cats = [t.category for t, _ in results[:3]]
        return Counter(cats).most_common(1)[0][0]

    def _display(self, output):
        print("=" * 70)
        print(f"New Ticket: {output['query_subject']}")
        print(f"Predicted Category: {output['category_prediction']}")
        print("=" * 70)

        level = "HIGH" if output["confidence"] >= 0.7 else (
            "MEDIUM" if output["confidence"] >= 0.4 else "LOW")
        bar = "#" * int(output["confidence"] * 25)
        print(f"Confidence: {output['confidence']:.3f} ({level}) |{bar}|")

        print(f"\nSimilar Past Tickets ({len(output['similar_tickets'])} found):")
        for i, sim_t in enumerate(output["similar_tickets"][:3], 1):
            print(f"\n  [{i}] {sim_t['ticket_id']}: {sim_t['subject']}")
            print(f"      Category: {sim_t['category']} | "
                  f"Priority: {sim_t['priority']} | "
                  f"Similarity: {sim_t['similarity']:.1%}")
            res_preview = sim_t["resolution"][:120]
            print(f"      Resolution: {res_preview}...")
            print(f"      Resolved in: {sim_t['resolution_time_hours']}h")

        print(f"\nSuggested Resolution:")
        for line in output["suggested_resolution"].split("\n"):
            if line.strip():
                wrapped = textwrap.fill(line.strip(), width=66,
                                        initial_indent="  ",
                                        subsequent_indent="  ")
                print(wrapped)

        print("=" * 70)


assistant = TicketAssistant(ticket_index)
print("Support Ticket Knowledge Assistant ready.")

Support Ticket Knowledge Assistant ready.


## Query Demonstrations

In [10]:
# ── Demo 1: Login problem ──
assistant.find_similar(
    subject="Can't access my account",
    description="I keep getting an error when trying to sign in. My password "
                "seems correct but the system rejects it every time."
)

New Ticket: Can't access my account
Predicted Category: Authentication
Confidence: 1.000 (HIGH) |#########################|

Similar Past Tickets (5 found):

  [1] TK-001: Cannot log in to dashboard
      Category: Authentication | Priority: high | Similarity: 68.9%
      Resolution: The user's account was locked after 5 failed login attempts due to the new security policy. Unlocked the account via adm...
      Resolved in: 2h

  [2] TK-006: Account shows as deactivated
      Category: Authentication | Priority: high | Similarity: 53.7%
      Resolution: The account was auto-deactivated due to 180 days of inactivity (company policy). Reactivated the account, restored all d...
      Resolved in: 2h

  [3] TK-005: Password reset email never arrives
      Category: Authentication | Priority: medium | Similarity: 50.7%
      Resolution: The user's email domain had strict DMARC policies that were rejecting our password reset emails. Added the domain to our...
      Resolved in: 3h

Suggeste

{'query_subject': "Can't access my account",
 'similar_tickets': [{'ticket_id': 'TK-001',
   'subject': 'Cannot log in to dashboard',
   'category': 'Authentication',
   'priority': 'high',
   'similarity': 0.689,
   'resolution': "The user's account was locked after 5 failed login attempts due to the new security policy. Unlocked the account via admin panel and advised the user to reset password using the 'Forgot Password' link. Also confirmed the email address was correct (user was using an old email).",
   'resolution_time_hours': 2},
  {'ticket_id': 'TK-006',
   'subject': 'Account shows as deactivated',
   'category': 'Authentication',
   'priority': 'high',
   'similarity': 0.537,
   'resolution': 'The account was auto-deactivated due to 180 days of inactivity (company policy). Reactivated the account, restored all data, and confirmed that the subscription billing was still active. Added an exception to prevent future auto-deactivation for paid accounts.',
   'resolution_time_hou

In [11]:
# ── Demo 2: Double billing ──
assistant.find_similar(
    subject="Duplicate payment on my card",
    description="There are two identical charges from your company on my credit "
                "card statement for this month. I should only have one charge."
)

New Ticket: Duplicate payment on my card
Predicted Category: Billing
Confidence: 1.000 (HIGH) |#########################|

Similar Past Tickets (5 found):

  [1] TK-007: Charged twice for subscription
      Category: Billing | Priority: high | Similarity: 79.2%
      Resolution: Confirmed duplicate charge in the billing system. The second charge was caused by a timeout during payment processing, w...
      Resolved in: 1h

  [2] TK-012: Usage overage charges unexpectedly high
      Category: Billing | Priority: high | Similarity: 46.8%
      Resolution: The customer exceeded their API call limit (100K/month) by 5x due to a misconfigured automation script running in a loop...
      Resolved in: 2h

  [3] TK-011: Requesting refund for unused months
      Category: Billing | Priority: high | Similarity: 44.2%
      Resolution: Investigation showed that the cancellation was submitted but not processed due to a system error in the cancellation wor...
      Resolved in: 3h

Suggested Resolut

{'query_subject': 'Duplicate payment on my card',
 'similar_tickets': [{'ticket_id': 'TK-007',
   'subject': 'Charged twice for subscription',
   'category': 'Billing',
   'priority': 'high',
   'similarity': 0.792,
   'resolution': 'Confirmed duplicate charge in the billing system. The second charge was caused by a timeout during payment processing, which triggered a retry. Issued a full refund for the duplicate charge ($49.99). Refund will appear in 5-7 business days. Added idempotency check to prevent future duplicates.',
   'resolution_time_hours': 1},
  {'ticket_id': 'TK-012',
   'subject': 'Usage overage charges unexpectedly high',
   'category': 'Billing',
   'priority': 'high',
   'similarity': 0.468,
   'resolution': 'The customer exceeded their API call limit (100K/month) by 5x due to a misconfigured automation script running in a loop. Issued a one-time courtesy credit of $250 (50% of overage). Set up usage alerts at 80% and 100% thresholds. Recommended the customer review t

In [12]:
# ── Demo 3: Slow performance ──
assistant.find_similar(
    subject="Everything is running extremely slow",
    description="Our whole platform is sluggish today. API calls that normally "
                "take milliseconds are now taking several seconds. "
                "This is impacting our production system."
)

New Ticket: Everything is running extremely slow
Predicted Category: Performance
Confidence: 1.000 (HIGH) |#########################|

Similar Past Tickets (5 found):

  [1] TK-016: API response times degraded
      Category: Performance | Priority: critical | Similarity: 67.8%
      Resolution: Root cause was a database connection pool exhaustion on the shared infrastructure cluster. A neighboring tenant's bulk i...
      Resolved in: 1h

  [2] TK-013: Dashboard loading very slowly
      Category: Performance | Priority: high | Similarity: 56.4%
      Resolution: The dashboard query was doing a full table scan on the records table due to a missing index on the created_at column. Th...
      Resolved in: 4h

  [3] TK-004: API key returning 401 unauthorized
      Category: Authentication | Priority: critical | Similarity: 41.2%
      Resolution: The API key had been automatically rotated as part of a scheduled security rotation that the account admin had configure...
      Resolved in: 

{'query_subject': 'Everything is running extremely slow',
 'similar_tickets': [{'ticket_id': 'TK-016',
   'subject': 'API response times degraded',
   'category': 'Performance',
   'priority': 'critical',
   'similarity': 0.678,
   'resolution': "Root cause was a database connection pool exhaustion on the shared infrastructure cluster. A neighboring tenant's bulk import job consumed excessive connections. Isolated the affected tenant, increased the connection pool size, and implemented per-tenant connection limits. Response times returned to normal within 15 minutes of the fix.",
   'resolution_time_hours': 1},
  {'ticket_id': 'TK-013',
   'subject': 'Dashboard loading very slowly',
   'category': 'Performance',
   'priority': 'high',
   'similarity': 0.564,
   'resolution': "The dashboard query was doing a full table scan on the records table due to a missing index on the created_at column. This became noticeable after the customer's data grew past 40K records. Added a composite index

In [13]:
# ── Demo 4: Integration issue ──
assistant.find_similar(
    subject="Notifications to our chat tool stopped",
    description="We have an integration that sends alerts to our team chat "
                "when files are uploaded. It stopped working a few days ago "
                "but the settings still show it as connected."
)

New Ticket: Notifications to our chat tool stopped
Predicted Category: Integration
Confidence: 1.000 (HIGH) |#########################|

Similar Past Tickets (3 found):

  [1] TK-019: Slack integration stopped sending notifications
      Category: Integration | Priority: medium | Similarity: 70.7%
      Resolution: The Slack webhook URL had expired because the Slack app was reinstalled in the customer's workspace, generating a new we...
      Resolved in: 1h

  [2] TK-020: Webhook events not being delivered
      Category: Integration | Priority: high | Similarity: 44.6%
      Resolution: The webhook was configured with event types using the old naming convention (file.uploaded) instead of the v2 format (fi...
      Resolved in: 2h

  [3] TK-014: File uploads timing out
      Category: Performance | Priority: medium | Similarity: 32.0%
      Resolution: The upload endpoint had a 60-second timeout that was too short for large files on the shared upload server. Increased th...
      Reso

{'query_subject': 'Notifications to our chat tool stopped',
 'similar_tickets': [{'ticket_id': 'TK-019',
   'subject': 'Slack integration stopped sending notifications',
   'category': 'Integration',
   'priority': 'medium',
   'similarity': 0.707,
   'resolution': "The Slack webhook URL had expired because the Slack app was reinstalled in the customer's workspace, generating a new webhook URL. The old URL was still configured in our system. Guided the customer to reconnect the integration (Settings > Integrations > Slack > Reconnect), which generated a fresh webhook URL. Notifications resumed immediately.",
   'resolution_time_hours': 1},
  {'ticket_id': 'TK-020',
   'subject': 'Webhook events not being delivered',
   'category': 'Integration',
   'priority': 'high',
   'similarity': 0.446,
   'resolution': 'The webhook was configured with event types using the old naming convention (file.uploaded) instead of the v2 format (files.upload.completed). Updated the webhook subscription to 

In [14]:
# ── Demo 5: Data issue ──
assistant.find_similar(
    subject="Characters appear garbled after CSV import",
    description="We uploaded a CSV with international characters (German umlauts, "
                "French accents) and they all appear as weird symbols in the system."
)

New Ticket: Characters appear garbled after CSV import
Predicted Category: Data
Confidence: 0.900 (HIGH) |######################|

Similar Past Tickets (2 found):

  [1] TK-026: Imported data showing special characters as garbled text
      Category: Data | Priority: medium | Similarity: 83.7%
      Resolution: The CSV file was encoded in UTF-8 with BOM but the import parser was interpreting it as Latin-1. Added UTF-8 BOM detecti...
      Resolved in: 3h

  [2] TK-025: Data export contains wrong date format
      Category: Data | Priority: low | Similarity: 41.2%
      Resolution: Added a date format option to the export settings. The customer can now choose between ISO 8601 (YYYY-MM-DD), US format ...
      Resolved in: 2h

Suggested Resolution:
  Based on similar ticket TK-026 (similarity: 84%):
  The CSV file was encoded in UTF-8 with BOM but the import parser
  was interpreting it as Latin-1. Added UTF-8 BOM detection to the
  import pipeline. Also added a character encoding select

{'query_subject': 'Characters appear garbled after CSV import',
 'similar_tickets': [{'ticket_id': 'TK-026',
   'subject': 'Imported data showing special characters as garbled text',
   'category': 'Data',
   'priority': 'medium',
   'similarity': 0.837,
   'resolution': 'The CSV file was encoded in UTF-8 with BOM but the import parser was interpreting it as Latin-1. Added UTF-8 BOM detection to the import pipeline. Also added a character encoding selector to the import wizard with auto-detection. Re-ran the import with correct encoding and all special characters displayed properly.',
   'resolution_time_hours': 3},
  {'ticket_id': 'TK-025',
   'subject': 'Data export contains wrong date format',
   'category': 'Data',
   'priority': 'low',
   'similarity': 0.412,
   'resolution': 'Added a date format option to the export settings. The customer can now choose between ISO 8601 (YYYY-MM-DD), US format (MM/DD/YYYY), or EU format (DD/MM/YYYY) in Settings > Export > Date Format. Default cha

In [15]:
# ── Demo 6: Category-filtered search ──
assistant.find_similar(
    subject="Payment issue",
    description="Something wrong with our billing. Need help.",
    category_filter="Billing"
)

New Ticket: Payment issue
Predicted Category: Billing
Confidence: 0.921 (HIGH) |#######################|

Similar Past Tickets (5 found):

  [1] TK-008: Cannot update payment method
      Category: Billing | Priority: medium | Similarity: 50.6%
      Resolution: The billing page JavaScript was blocked by the user's ad blocker (uBlock Origin). The Stripe payment iframe was being fi...
      Resolved in: 1h

  [2] TK-012: Usage overage charges unexpectedly high
      Category: Billing | Priority: high | Similarity: 45.6%
      Resolution: The customer exceeded their API call limit (100K/month) by 5x due to a misconfigured automation script running in a loop...
      Resolved in: 2h

  [3] TK-007: Charged twice for subscription
      Category: Billing | Priority: high | Similarity: 44.4%
      Resolution: Confirmed duplicate charge in the billing system. The second charge was caused by a timeout during payment processing, w...
      Resolved in: 1h

Suggested Resolution:
  Based on simila

{'query_subject': 'Payment issue',
 'similar_tickets': [{'ticket_id': 'TK-008',
   'subject': 'Cannot update payment method',
   'category': 'Billing',
   'priority': 'medium',
   'similarity': 0.506,
   'resolution': "The billing page JavaScript was blocked by the user's ad blocker (uBlock Origin). The Stripe payment iframe was being filtered. Solution: asked user to disable ad blocker for our domain or use the direct billing URL (/settings/billing/update-card). Also filed a bug to add a fallback UI when the iframe is blocked.",
   'resolution_time_hours': 1},
  {'ticket_id': 'TK-012',
   'subject': 'Usage overage charges unexpectedly high',
   'category': 'Billing',
   'priority': 'high',
   'similarity': 0.456,
   'resolution': 'The customer exceeded their API call limit (100K/month) by 5x due to a misconfigured automation script running in a loop. Issued a one-time courtesy credit of $250 (50% of overage). Set up usage alerts at 80% and 100% thresholds. Recommended the customer rev

## Resolution Pattern Analysis

Let's analyze the resolved ticket database to find common resolution patterns.

In [16]:
# ── Resolution pattern analysis ──

print("=== Resolution Patterns by Category ===\n")
for category in sorted(set(t.category for t in tickets)):
    cat_tickets = [t for t in tickets if t.category == category]
    avg_time = np.mean([t.resolution_time for t in cat_tickets])
    priorities = Counter(t.priority for t in cat_tickets)

    print(f"{category} ({len(cat_tickets)} tickets)")
    print(f"  Avg resolution time: {avg_time:.1f} hours")
    print(f"  Priority distribution: {dict(priorities)}")

    # Find common words in resolutions
    all_res_words = " ".join(t.resolution.lower() for t in cat_tickets)
    common_actions = []
    action_patterns = [
        ("updated", "configuration update"),
        ("reset", "reset/restart"),
        ("added", "added setting/feature"),
        ("fixed", "bug fix"),
        ("increased", "capacity increase"),
        ("refund", "refund issued"),
        ("restored", "data restoration"),
        ("guided", "user guidance"),
    ]
    for word, label in action_patterns:
        if word in all_res_words:
            common_actions.append(label)

    print(f"  Common fix types: {', '.join(common_actions[:4])}")
    print()

=== Resolution Patterns by Category ===

Authentication (6 tickets)
  Avg resolution time: 2.2 hours
  Priority distribution: {'high': 3, 'critical': 2, 'medium': 1}
  Common fix types: configuration update, reset/restart, added setting/feature, data restoration

Billing (6 tickets)
  Avg resolution time: 2.2 hours
  Priority distribution: {'high': 3, 'medium': 2, 'low': 1}
  Common fix types: configuration update, added setting/feature, bug fix, refund issued

Data (6 tickets)
  Avg resolution time: 3.2 hours
  Priority distribution: {'low': 2, 'medium': 1, 'critical': 2, 'high': 1}
  Common fix types: configuration update, added setting/feature, capacity increase, data restoration

Integration (6 tickets)
  Avg resolution time: 2.2 hours
  Priority distribution: {'medium': 3, 'high': 3}
  Common fix types: configuration update, user guidance

Performance (6 tickets)
  Avg resolution time: 3.5 hours
  Priority distribution: {'high': 3, 'medium': 2, 'critical': 1}
  Common fix types: a

## Evaluation

In [17]:
# ── Evaluation set — paraphrased versions of real tickets ──

EVAL_SET = [
    {"query": "Sign-in not working, says wrong password every time",
     "expected": "TK-001", "category": "Authentication"},
    {"query": "Our enterprise SSO broke after we rebranded",
     "expected": "TK-002", "category": "Authentication"},
    {"query": "TOTP codes from authenticator app always invalid",
     "expected": "TK-003", "category": "Authentication"},
    {"query": "Billed the same amount twice on my statement",
     "expected": "TK-007", "category": "Billing"},
    {"query": "Can't change the credit card in the settings page",
     "expected": "TK-008", "category": "Billing"},
    {"query": "Main page takes ages to load, was fast before",
     "expected": "TK-013", "category": "Performance"},
    {"query": "Large file uploads fail with timeout",
     "expected": "TK-014", "category": "Performance"},
    {"query": "Slack alerts for uploads stopped coming through",
     "expected": "TK-019", "category": "Integration"},
    {"query": "CSV import garbles international characters",
     "expected": "TK-026", "category": "Data"},
    {"query": "Automated data backups haven't run in a week",
     "expected": "TK-030", "category": "Data"},
]

# Evaluate both approaches
kw_recall_1 = 0
kw_recall_3 = 0
sem_recall_1 = 0
sem_recall_3 = 0
kw_mrr_vals = []
sem_mrr_vals = []

print(f"{'Query':<50} {'KW@1':^7} {'Sem@1':^7}")
print("-" * 65)

for item in EVAL_SET:
    q = item["query"]
    expected = item["expected"]

    kw = keyword_search(q, tickets, top_k=5)
    kw_ids = [t.ticket_id for t, _ in kw]
    kw_hit1 = kw_ids[0] == expected if kw_ids else False
    kw_hit3 = expected in kw_ids[:3]
    kw_recall_1 += int(kw_hit1)
    kw_recall_3 += int(kw_hit3)
    kw_rank = (kw_ids.index(expected) + 1) if expected in kw_ids else 0
    kw_mrr_vals.append(1.0 / kw_rank if kw_rank > 0 else 0)

    sem = ticket_index.search(q, top_k=5)
    sem_ids = [t.ticket_id for t, _ in sem]
    sem_hit1 = sem_ids[0] == expected if sem_ids else False
    sem_hit3 = expected in sem_ids[:3]
    sem_recall_1 += int(sem_hit1)
    sem_recall_3 += int(sem_hit3)
    sem_rank = (sem_ids.index(expected) + 1) if expected in sem_ids else 0
    sem_mrr_vals.append(1.0 / sem_rank if sem_rank > 0 else 0)

    print(f"  {q[:48]:<48} {'HIT' if kw_hit1 else 'MISS':^7} "
          f"{'HIT' if sem_hit1 else 'MISS':^7}")

n = len(EVAL_SET)
print("-" * 65)
print(f"\n{'Metric':<25} {'Keyword':^12} {'Semantic':^12}")
print("-" * 50)
print(f"  {'Recall@1':<23} {kw_recall_1/n:^12.1%} {sem_recall_1/n:^12.1%}")
print(f"  {'Recall@3':<23} {kw_recall_3/n:^12.1%} {sem_recall_3/n:^12.1%}")
print(f"  {'MRR':<23} {np.mean(kw_mrr_vals):^12.3f} {np.mean(sem_mrr_vals):^12.3f}")

Query                                               KW@1    Sem@1 
-----------------------------------------------------------------
  Sign-in not working, says wrong password every t   HIT     HIT  
  Our enterprise SSO broke after we rebranded        HIT     HIT  


  TOTP codes from authenticator app always invalid   HIT     HIT  


  Billed the same amount twice on my statement       HIT     HIT  


  Can't change the credit card in the settings pag   HIT     HIT  
  Main page takes ages to load, was fast before      HIT     HIT  
  Large file uploads fail with timeout               HIT     HIT  


  Slack alerts for uploads stopped coming through    HIT     HIT  


  CSV import garbles international characters        HIT     HIT  


  Automated data backups haven't run in a week       HIT     HIT  
-----------------------------------------------------------------

Metric                      Keyword      Semantic  
--------------------------------------------------
  Recall@1                   100.0%       100.0%   
  Recall@3                   100.0%       100.0%   
  MRR                        1.000        1.000    


In [18]:
# ── Category prediction accuracy ──

print("=== Category Prediction from Similar Tickets ===\n")
cat_correct = 0
for item in EVAL_SET:
    q = item["query"]
    expected_cat = item["category"]

    results = ticket_index.search(q, top_k=3)
    predicted_cats = [t.category for t, _ in results]
    predicted = Counter(predicted_cats).most_common(1)[0][0]

    ok = predicted == expected_cat
    cat_correct += int(ok)
    status = "OK" if ok else "MISS"
    print(f"  [{status}] \"{q[:45]}\"")
    print(f"         Expected: {expected_cat} | Predicted: {predicted}")

print(f"\nCategory prediction accuracy: {cat_correct}/{n} = {cat_correct/n:.1%}")

=== Category Prediction from Similar Tickets ===

  [OK] "Sign-in not working, says wrong password ever"
         Expected: Authentication | Predicted: Authentication


  [OK] "Our enterprise SSO broke after we rebranded"
         Expected: Authentication | Predicted: Authentication


  [OK] "TOTP codes from authenticator app always inva"
         Expected: Authentication | Predicted: Authentication
  [OK] "Billed the same amount twice on my statement"
         Expected: Billing | Predicted: Billing


  [OK] "Can't change the credit card in the settings "


         Expected: Billing | Predicted: Billing
  [OK] "Main page takes ages to load, was fast before"
         Expected: Performance | Predicted: Performance
  [OK] "Large file uploads fail with timeout"
         Expected: Performance | Predicted: Performance
  [OK] "Slack alerts for uploads stopped coming throu"
         Expected: Integration | Predicted: Integration


  [OK] "CSV import garbles international characters"
         Expected: Data | Predicted: Data


  [OK] "Automated data backups haven't run in a week"
         Expected: Data | Predicted: Data

Category prediction accuracy: 10/10 = 100.0%


## Error Analysis

In [19]:
# ── Edge cases and failure modes ──

edge_cases = [
    ("Feature request: add dark mode to the dashboard",
     "feature_request"),
    ("Hello, I just wanted to say your product is great!",
     "not_a_problem"),
    ("My computer is making a weird noise",
     "out_of_scope"),
    ("Need help urgently!!!!",
     "vague_no_detail"),
]

print("=== Edge Case Analysis ===\n")
for query, case_type in edge_cases:
    results = ticket_index.search(query, top_k=2)
    top_score = results[0][1] if results else 0

    print(f"Q: \"{query}\"")
    print(f"  Case type: {case_type}")
    print(f"  Top similarity: {top_score:.3f}")
    if top_score < SIMILARITY_THRESHOLD:
        print(f"  -> BELOW THRESHOLD ({SIMILARITY_THRESHOLD}): "
              f"No confident match. Flag for manual review.")
    else:
        print(f"  -> Matched: {results[0][0].summary}")
    print()

=== Edge Case Analysis ===

Q: "Feature request: add dark mode to the dashboard"
  Case type: feature_request
  Top similarity: 0.215
  -> BELOW THRESHOLD (0.3): No confident match. Flag for manual review.

Q: "Hello, I just wanted to say your product is great!"
  Case type: not_a_problem
  Top similarity: 0.112
  -> BELOW THRESHOLD (0.3): No confident match. Flag for manual review.

Q: "My computer is making a weird noise"
  Case type: out_of_scope
  Top similarity: 0.149
  -> BELOW THRESHOLD (0.3): No confident match. Flag for manual review.



Q: "Need help urgently!!!!"
  Case type: vague_no_detail
  Top similarity: 0.137
  -> BELOW THRESHOLD (0.3): No confident match. Flag for manual review.



In [20]:
# ── Export metrics ──

metrics = {
    "project": "Support Ticket Knowledge Assistant",
    "knowledge_base": {
        "total_tickets": len(RESOLVED_TICKETS),
        "categories": dict(Counter(t["category"] for t in RESOLVED_TICKETS)),
        "priorities": dict(Counter(t["priority"] for t in RESOLVED_TICKETS)),
        "avg_resolution_hours": float(np.mean(
            [t["resolution_time_hours"] for t in RESOLVED_TICKETS])),
    },
    "retrieval_backend": ticket_index.backend,
    "evaluation": {
        "eval_queries": len(EVAL_SET),
        "keyword": {
            "recall_at_1": kw_recall_1 / n,
            "recall_at_3": kw_recall_3 / n,
            "mrr": float(np.mean(kw_mrr_vals)),
        },
        "semantic": {
            "recall_at_1": sem_recall_1 / n,
            "recall_at_3": sem_recall_3 / n,
            "mrr": float(np.mean(sem_mrr_vals)),
        },
        "category_prediction_accuracy": cat_correct / n,
    },
    "config": {
        "top_k": TOP_K,
        "similarity_threshold": SIMILARITY_THRESHOLD,
        "embedding_model": EMBEDDING_MODEL if USE_ST else "TF-IDF",
    },
}

metrics_path = os.path.join(SAVE_DIR, "metrics.json")
with open(metrics_path, "w") as f:
    json.dump(metrics, f, indent=2)
print(f"Metrics saved to {metrics_path}")
print(json.dumps(metrics, indent=2))

Metrics saved to E:\Github\Machine-Learning-Projects\GenAI\17_Support_Ticket_Knowledge_Assistant\metrics.json
{
  "project": "Support Ticket Knowledge Assistant",
  "knowledge_base": {
    "total_tickets": 30,
    "categories": {
      "Authentication": 6,
      "Billing": 6,
      "Performance": 6,
      "Integration": 6,
      "Data": 6
    },
    "priorities": {
      "high": 13,
      "critical": 5,
      "medium": 9,
      "low": 3
    },
    "avg_resolution_hours": 2.6333333333333333
  },
  "retrieval_backend": "chroma",
  "evaluation": {
    "eval_queries": 10,
    "keyword": {
      "recall_at_1": 1.0,
      "recall_at_3": 1.0,
      "mrr": 1.0
    },
    "semantic": {
      "recall_at_1": 1.0,
      "recall_at_3": 1.0,
      "mrr": 1.0
    },
    "category_prediction_accuracy": 1.0
  },
  "config": {
    "top_k": 5,
    "similarity_threshold": 0.3,
    "embedding_model": "all-MiniLM-L6-v2"
  }
}


## Limitations

1. **No LLM answer generation** — resolutions are copied from past tickets
   verbatim. A production system would use an LLM to adapt the resolution
   to the new ticket's specific context.

2. **Small knowledge base** — 30 tickets is tiny. Real systems have 100K+
   tickets, requiring scalable indexing and approximate nearest neighbors.

3. **No resolution quality tracking** — we don't know which past resolutions
   actually worked. Incorporating customer satisfaction scores would improve
   suggestion quality.

4. **No temporal awareness** — old resolutions may be outdated. A ticket
   from 2 years ago about a fixed bug shouldn't be suggested for new issues.

5. **No ticket threading** — real tickets have conversations (back-and-forth).
   We only use the initial description, missing follow-up context.

6. **No personalization** — the system doesn't consider the customer's account
   type, plan, or history, which would help narrow the resolution.

## Common Mistakes

1. **Over-relying on keyword search** — keyword matching misses paraphrased
   descriptions. Always compare against semantic search for text-heavy use cases.

2. **Ignoring similarity threshold** — returning the "most similar" ticket even
   when similarity is 0.05. Low-similarity results are noise, not matches.
   Set a minimum threshold (e.g., 0.3) and say "no similar tickets" when nothing
   meets it.

3. **Suggesting outdated resolutions** — a 3-year-old resolution for a
   since-patched bug is harmful. Weight recent tickets higher.

4. **Not using category as a filter** — if the user already selected "Billing,"
   searching across all categories adds noise. Use category metadata.

5. **Matching on priority, not content** — two "critical" tickets are not
   necessarily related. Priority is metadata, not content. It should filter,
   not drive similarity.

6. **Single-ticket suggestions** — showing only the #1 match is fragile. Show
   top 3 so the agent can choose the most relevant resolution.

7. **Not explaining *why* tickets are similar** — agents need to see the
   connection. Highlighting shared terms or topics builds trust.

## Mini Challenge

### Exercise 1: Hybrid Search
Implement a hybrid that combines keyword and semantic scores:
```python
hybrid_score = alpha * semantic_score + (1 - alpha) * keyword_score
```
Test with alpha values from 0.0 to 1.0 and find the optimal balance.

### Exercise 2: Resolution Clustering
Group the 30 resolutions into clusters (e.g., "configuration fix,"
"bug fix," "user guidance"). When suggesting a fix, also show which
fix *type* is most common for this kind of problem.

### Exercise 3: Duplicate Detection
Given two new tickets, determine if they describe the same problem
(similarity > 0.8). Build a deduplication function.

### Exercise 4: Priority Prediction
Based on similar past tickets' priorities, predict the priority of
a new ticket. Evaluate accuracy.

### Exercise 5: Resolution Template Generator
From the top 3 most similar tickets, extract common resolution steps
and generate a resolution template that an agent can customize.

## Production Considerations

1. **Scale** — use Faiss, Milvus, or Pinecone for indexing 100K+ tickets
   with sub-second search latency.

2. **Incremental updates** — new resolved tickets should be indexed
   incrementally, not requiring a full rebuild.

3. **Feedback loop** — track whether suggested resolutions are accepted
   or modified. Use this to re-rank future suggestions.

4. **LLM synthesis** — use an LLM to adapt past resolutions to the new
   ticket's specific context (different account, slightly different symptoms).

5. **Agent UI** — embed in the support agent's dashboard (Zendesk, ServiceNow)
   as a sidebar that shows similar tickets as the agent reads the new one.

6. **Self-service** — expose to end users as a "find similar issues" search
   before they file a ticket. Can deflect 20-40% of incoming tickets.

7. **Ticket routing** — use the category prediction to auto-route tickets
   to the right support team, reducing triage time.

8. **Analytics** — track top unmatched queries (no similar tickets found)
   to identify knowledge gaps in the ticket database.

## How to Improve This Project

- **Hybrid search** — combine BM25 (keyword) with semantic embeddings for
  best-of-both-worlds retrieval.
- **Fine-tuned embeddings** — train on your actual ticket pairs to learn
  domain-specific similarity (e.g., SaaS terminology).
- **Resolution scoring** — weight tickets by customer satisfaction score
  or reopening rate. Only suggest resolutions that actually worked.
- **Time decay** — exponentially down-weight old tickets so recent
  resolutions are preferred over stale ones.
- **Multi-turn context** — include the full ticket conversation thread,
  not just the initial description.
- **Cross-product search** — if your company has multiple products, share
  the knowledge base with product-level filtering.

## Key Takeaways

1. **Semantic search beats keyword search for support tickets** — users
   describe the same problem in many ways. Semantic similarity captures
   meaning; keyword matching misses synonyms and paraphrases.

2. **Keyword search still has value** — for exact error codes, API
   endpoints, and technical identifiers, keyword match is more precise.
   The best systems use both (hybrid).

3. **Similarity thresholds prevent false matches** — always set a minimum
   threshold. A low-similarity "match" is worse than saying "no match found."

4. **Category metadata improves precision** — filtering by ticket category
   before searching reduces noise significantly.

5. **Resolution suggestion needs context** — copying a past resolution
   verbatim is a starting point. Production systems adapt the suggestion
   to the new ticket's specifics.

6. **Evaluation requires paraphrased test queries** — the evaluation set
   must use different wording than the knowledge base to test true
   semantic understanding, not just memorization.